In [37]:
import pandas as pd
import glob

In [38]:
files = glob.glob('*.xlsx')
files

['shopify_1.xlsx',
 'shopify_10.xlsx',
 'shopify_2.xlsx',
 'shopify_3.xlsx',
 'shopify_4.xlsx',
 'shopify_5.xlsx',
 'shopify_6.xlsx',
 'shopify_7.xlsx',
 'shopify_8.xlsx',
 'shopify_9.xlsx']

In [39]:
df_list = []
for file in files:
    df = pd.read_excel(file)
    df_list.append(df)
full_df = pd.concat(df_list, ignore_index=True)

In [40]:
full_df.shape

(200000, 3)

In [41]:
print(full_df.head())
print(full_df.info())

             domain average_product_price  \
0  www.store1_0.com            USD $551.8   
1  www.store1_1.com          USD $1816.86   
2  www.store1_2.com          USD $1384.22   
3  www.store1_3.com          USD $1592.81   
4  www.store1_4.com          USD $1970.23   

                                  categories  
0                      /Toys & Hobbies/Dolls  
1         /Beauty & Fitness/Face & Body Care  
2           /Apparel/Footwear/Athletic Shoes  
3         /Beauty & Fitness/Face & Body Care  
4  /Food & Drink/Food/Baked Goods & Desserts  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 3 columns):
 #   Column                 Non-Null Count   Dtype 
---  ------                 --------------   ----- 
 0   domain                 200000 non-null  object
 1   average_product_price  200000 non-null  object
 2   categories             200000 non-null  object
dtypes: object(3)
memory usage: 4.6+ MB
None


## Spliting categories

In [42]:
# inspect one row
full_df['categories'].iloc[0]
full_df['categories'].iloc[0].split('/')

['', 'Toys & Hobbies', 'Dolls']

In [43]:
# apply and extract main category
full_df['main_category'] = full_df['categories'].str.split('/').str[1]
full_df['sub_category'] = full_df['categories'].str.split('/').str[2:4]
full_df[['categories', 'main_category', 'sub_category']].head(5)

,categories,main_category,sub_category
0,/Toys & Hobbies/Dolls,Toys & Hobbies,[Dolls]
1,/Beauty & Fitness/Face & Body Care,Beauty & Fitness,[Face & Body Care]
2,/Apparel/Footwear/Athletic Shoes,Apparel,"[Footwear, Athletic Shoes]"
3,/Beauty & Fitness/Face & Body Care,Beauty & Fitness,[Face & Body Care]
4,/Food & Drink/Food/Baked Goods & Desserts,Food & Drink,"[Food, Baked Goods & Desserts]"


In [44]:
full_df['main_category'].unique()

array(['Toys & Hobbies', 'Beauty & Fitness', 'Apparel', 'Food & Drink',
       'Home & Garden', 'Business & Industrial', 'Sports'], dtype=object)

In [45]:
full_df['main_category'].isnull().sum()

0

## Cleaning price column

In [46]:
full_df['average_product_price'].head()

0      USD $551.8
1    USD $1816.86
2    USD $1384.22
3    USD $1592.81
4    USD $1970.23
Name: average_product_price, dtype: object

In [47]:
full_df['average_product_price'] = (full_df['average_product_price'].str.replace(r"[^\d.]", "", regex=True).astype(float))
full_df['average_product_price'].head()

0     551.80
1    1816.86
2    1384.22
3    1592.81
4    1970.23
Name: average_product_price, dtype: float64

## Split dataset into separate file

In [48]:
categories = full_df['main_category'].unique()
for category in categories:
    category_df = full_df[full_df['main_category'] == category]
    safe_name = category.replace('&', '_')
    category_df.to_excel(f'{safe_name}_products.xlsx', index=False)